In [1]:
import cv2
import os

# --- 1. Fixed Input Logic ---
# Getting the input once and matching it to the correct paths
choice = input("Enter 'train', 'validation', 'augmented', or '3rd': ").strip().lower()

if choice == 'train':
    image_folder = "../data_Set/train/train/images"
    label_folder = "../data_Set/train/train/labels"
elif choice in ['validation', 'v']:
    image_folder = "../data_Set/validation/images"
    label_folder = "../data_Set/validation/labels"
elif choice in ['augmented', 'a']:
    image_folder = "../data_Set/train/augmented_images/images"
    label_folder = "../data_Set/train/augmented_images/labels"
elif choice == '3rd':
    image_folder = "../data_Set/train/3rd_train/images"
    label_folder = "../data_Set/train/3rd_train/labels"
else:
    print("Invalid input. Exiting.")
    exit()

# Ensure the label folder exists
os.makedirs(label_folder, exist_ok=True)

rects = []
drawing = False
start_pt = (0, 0)
current_img = None
clone = None

def draw_rectangle(event, x, y, flags, param):
    global rects, drawing, start_pt, current_img, clone
    if event == cv2.EVENT_LBUTTONDOWN:
        drawing = True
        start_pt = (x, y)
    elif event == cv2.EVENT_MOUSEMOVE and drawing:
        current_img = clone.copy()
        cv2.rectangle(current_img, start_pt, (x, y), (0, 255, 0), 2)
    elif event == cv2.EVENT_LBUTTONUP:
        drawing = False
        end_pt = (x, y)
        cv2.rectangle(current_img, start_pt, end_pt, (0, 255, 0), 2)
        rects.append((start_pt, end_pt))
        clone = current_img.copy()

cv2.namedWindow("Annotator")
cv2.setMouseCallback("Annotator", draw_rectangle)

# Get list of images
image_files = [f for f in os.listdir(image_folder) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]

for filename in image_files:
    img_path = os.path.join(image_folder, filename)
    label_path = os.path.join(
        label_folder,
        os.path.splitext(filename)[0] + ".txt"
    )

    # Skip images that already have labels
    if os.path.exists(label_path):
        print(f"Skipping {filename} (label already exists)")
        continue

    current_img = cv2.imread(img_path)
    if current_img is None:
        continue

    clone = current_img.copy()
    rects = []

    print(f"Labeling {filename}... [s] Save | [c] Clear | [d] Delete Image | [q] Quit")
    
    while True:
        cv2.imshow("Annotator", current_img)
        key = cv2.waitKey(1) & 0xFF
        
        # Save Annotations
        if key == ord('s'):
            h, w, _ = current_img.shape
            with open(label_path, "w") as f:
                for (pt1, pt2) in rects:
                    x1, y1 = pt1; x2, y2 = pt2
                    x_center = ((x1 + x2) / 2) / w
                    y_center = ((y1 + y2) / 2) / h
                    box_w = abs(x1 - x2) / w
                    box_h = abs(y1 - y2) / h
                    f.write(f"0 {x_center:.6f} {y_center:.6f} {box_w:.6f} {box_h:.6f}\n")
            print(f"Saved: {label_path}")
            break 
            
        # Clear current bounding boxes
        elif key == ord('c'):
            rects = []
            current_img = cv2.imread(img_path)
            clone = current_img.copy()
            
        # --- 2. Added Delete Functionality ---
        elif key == ord('d'):
            try:
                os.remove(img_path)
                print(f"Deleted image: {filename}")
            except Exception as e:
                print(f"Error deleting file {filename}: {e}")
            break # Skip to the next image since this one is gone
            
        # Quit script
        elif key == ord('q'):
            cv2.destroyAllWindows()
            exit()

cv2.destroyAllWindows()
print("All images processed.")

Skipping 00149629982.jpg (label already exists)
Skipping 00182743849.jpg (label already exists)
Skipping 00216324277.jpg (label already exists)
Skipping 00269846176.jpg (label already exists)
Skipping 00380966697.jpg (label already exists)
Skipping 01028518429.jpg (label already exists)
Skipping 01354311910.jpg (label already exists)
Skipping 01449100005.jpg (label already exists)
Skipping 01593351578.jpg (label already exists)
Skipping 01618949505.jpg (label already exists)
Skipping 01684390394.jpg (label already exists)
Skipping 01807130008.jpg (label already exists)
Skipping 01851395531.jpg (label already exists)
Skipping 01861430989.jpg (label already exists)
Skipping 01891151076.jpg (label already exists)
Skipping 02155578111.jpg (label already exists)
Skipping 02246053785.jpg (label already exists)
Skipping 02311440156.jpg (label already exists)
Skipping 02434920001.jpg (label already exists)
Skipping 02635059922.jpg (label already exists)
Skipping 02741151662.jpg (label already 